<a href="https://colab.research.google.com/github/ProgramNewbbie/FPGA-Transformer-Accelerator-Extension/blob/main/Extension_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import time
import threading
import queue

# [제어 공간]
free_buffer_queue = queue.Queue()
free_buffer_queue.put(0)
free_buffer_queue.put(1)

fpga_task_queue = queue.Queue()
fpga_done_queue = queue.Queue()
cpu_post_queue = queue.Queue()

# =====================================================================
# 💡 [가상 구현 1] 4개의 물리적 버퍼 공간 (Input 2개, Output 2개)
# =====================================================================
input_buffers = {0: None, 1: None}
output_buffers = {0: None, 1: None}
# [실제 보드 적용 시] -> pynq.allocate를 사용하여 입출력용 물리 메모리(CMA) 총 4개 할당

# =====================================================================
# 💡 [가상 구현 2] [Pipeline Stage 2] FPGA 하드웨어 연산 스테이지
# =====================================================================
def mock_fpga_kernel(buf_idx):
    in_data = input_buffers[buf_idx]
    time.sleep(0.5)
    output_buffers[buf_idx] = f"[{in_data}의 H/W 연산 완료]"
    # [실제 보드 적용 시] -> FPGA IP 레지스터(0x10, 0x1C 등)에 입출력 버퍼 주소 전달 후 AP_START(0x00, 0x01) 실행

def fpga_async_worker():
    while True:
        buf_idx = fpga_task_queue.get()
        if buf_idx is None: break
        mock_fpga_kernel(buf_idx)
        fpga_done_queue.put(buf_idx)
        fpga_task_queue.task_done()

# =====================================================================
# 💡 [가상 구현 3] 인터럽트 처리기
# =====================================================================
def interrupt_worker():
    while True:
        buf_idx = fpga_done_queue.get()
        if buf_idx is None: break
        cpu_post_queue.put(buf_idx)
        fpga_done_queue.task_done()
        # [실제 보드 적용 시] -> PYNQ의 my_interrupt.wait()를 사용해 하드웨어 완료 신호 대기

# =====================================================================
# 💡 [Pipeline Stage 1] 호스트 CPU 메인 스레드 (전처리/입력 & 결과/후처리)
# =====================================================================
def run_simulation(total_data_count):
    start_time = time.time()

    t1 = threading.Thread(target=fpga_async_worker, daemon=True)
    t2 = threading.Thread(target=interrupt_worker, daemon=True)
    t1.start()
    t2.start()

    for i in range(1, total_data_count + 1):
        buf_idx = None
        is_waiting = False

        while buf_idx is None:
            # [Stage 1 - 후처리] 연산 완료된 Output 결과 회수
            while not cpu_post_queue.empty():
                done_idx = cpu_post_queue.get()
                result_data = output_buffers[done_idx]
                print(f"[t={time.time()-start_time:.2f}s] 🧑‍💻 [Stage 1] Output {done_idx} '{result_data}' 후처리 완료")
                free_buffer_queue.put(done_idx)
                is_waiting = False

            # [Stage 1 - 전처리 대기] 빈 버퍼 티켓 획득
            try:
                buf_idx = free_buffer_queue.get_nowait()
            except queue.Empty:
                if not is_waiting:
                    is_waiting = True
                time.sleep(0.01)

        # [Stage 1 - 전처리/입력] Input 버퍼에 데이터 채우고 Stage 2로 넘김
        input_buffers[buf_idx] = f"데이터 {i}"
        print(f"[t={time.time()-start_time:.2f}s] 🧑‍💻 [Stage 1] Input {buf_idx}에 데이터 {i} 입력 -> FPGA 전달")
        fpga_task_queue.put(buf_idx)

    while free_buffer_queue.qsize() < 2:
        if not cpu_post_queue.empty():
            done_idx = cpu_post_queue.get()
            print(f"[t={time.time()-start_time:.2f}s] 🧑‍💻 [Stage 1] Output {done_idx} 최종 후처리 완료")
            free_buffer_queue.put(done_idx)
        time.sleep(0.01)

    fpga_task_queue.put(None)
    fpga_done_queue.put(None)
    t1.join()
    t2.join()

if __name__ == "__main__":
    run_simulation(4)
